# GDELT 2026-09-13 중동 국가 간 선택 EventCode 분석

## 분석 목적
1. `Actor1CountryCode`와 `Actor2CountryCode`가 **모두 지정한 중동 16개국**에 속하는 이벤트를 필터링합니다.
2. 그중 지정한 20개 `EventCode`만 필터링합니다.
3. `Actor1 → Actor2` 방향을 유지하여 **국가쌍별 이벤트 수**를 집계합니다.

> `Actor1CountryCode`, `Actor2CountryCode`는 ISO 2자리 코드가 아니라 GDELT/CAMEO의 3자리 국가 코드를 사용합니다.  
> `EventCode`는 `1381`처럼 계층형 코드이며 앞자리 0이 존재할 수 있으므로 문자열로 다루는 것이 안전합니다.

In [16]:
import pandas as pd

# 경로명은 로컬환경에 맞게 수정하여 사용해주세요.
file_path = "data/20260913.export.CSV"

## 1. GDELT 1.0 Daily Event 파일 불러오기

In [2]:
headers = [
    "GLOBALEVENTID","SQLDATE","MonthYear","Year","FractionDate",
    "Actor1Code","Actor1Name","Actor1CountryCode","Actor1KnownGroupCode","Actor1EthnicCode",
    "Actor1Religion1Code","Actor1Religion2Code","Actor1Type1Code","Actor1Type2Code","Actor1Type3Code",
    "Actor2Code","Actor2Name","Actor2CountryCode","Actor2KnownGroupCode","Actor2EthnicCode",
    "Actor2Religion1Code","Actor2Religion2Code","Actor2Type1Code","Actor2Type2Code","Actor2Type3Code",
    "IsRootEvent","EventCode","EventBaseCode","EventRootCode","QuadClass","GoldsteinScale",
    "NumMentions","NumSources","NumArticles","AvgTone",
    "Actor1Geo_Type","Actor1Geo_FullName","Actor1Geo_CountryCode","Actor1Geo_ADM1Code",
    "Actor1Geo_Lat","Actor1Geo_Long","Actor1Geo_FeatureID",
    "Actor2Geo_Type","Actor2Geo_FullName","Actor2Geo_CountryCode","Actor2Geo_ADM1Code",
    "Actor2Geo_Lat","Actor2Geo_Long","Actor2Geo_FeatureID",
    "ActionGeo_Type","ActionGeo_FullName","ActionGeo_CountryCode","ActionGeo_ADM1Code",
    "ActionGeo_Lat","ActionGeo_Long","ActionGeo_FeatureID",
    "DATEADDED","SOURCEURL"
]

df = pd.read_csv(
    file_path,
    sep="\t",
    header=None,
    names=headers,
    dtype=str
)

print("원본 데이터 크기:", df.shape)
df.head()

원본 데이터 크기: (60559, 58)


,GLOBALEVENTID,SQLDATE,MonthYear,Year,FractionDate,Actor1Code,Actor1Name,Actor1CountryCode,Actor1KnownGroupCode,Actor1EthnicCode,...,Actor2Geo_FeatureID,ActionGeo_Type,ActionGeo_FullName,ActionGeo_CountryCode,ActionGeo_ADM1Code,ActionGeo_Lat,ActionGeo_Long,ActionGeo_FeatureID,DATEADDED,SOURCEURL
0,1322780368,20250913,202509,2025,2025.6932,NaN,NaN,NaN,NaN,NaN,...,-1150871,4,"Alba Iulia, Alba, Romania",RO,RO01,46.0667,23.5833,-1150871,20260913,https://www.timesargus.com/news/world/romania-...
1,1322780369,20250913,202509,2025,2025.6932,CHN,CHINA,CHN,NaN,NaN,...,NaN,4,"Shanghai, Shanghai, China",CH,CH23,31.2222,121.458,-1924465,20260913,http://europe.chinadaily.com.cn/a/202609/13/WS...
2,1322780370,20250913,202509,2025,2025.6932,GOV,PRESIDENT,NaN,NaN,NaN,...,-1924465,4,"Shanghai, Shanghai, China",CH,CH23,31.2222,121.458,-1924465,20260913,http://europe.chinadaily.com.cn/a/202609/13/WS...
3,1322780371,20250913,202509,2025,2025.6932,ROU,ROMANIA,NaN,NaN,NaN,...,NaN,4,"Alba Iulia, Alba, Romania",RO,RO01,46.0667,23.5833,-1150871,20260913,https://www.timesargus.com/news/world/romania-...
4,1322780372,20250913,202509,2025,2025.6932,USA,UNITED STATES,USA,NaN,NaN,...,NaN,3,"Manhattan, New York, United States",US,USNY,40.7834,-73.9662,971945,20260913,https://www.shockya.com/news/2026/09/12/jay-z-...


## 2. 중동 16개국 정의 및 Actor1·Actor2 필터링

In [3]:
middle_east_cameo = {
    "SAU": "사우디아라비아",
    "UAE": "아랍에미리트",
    "QAT": "카타르",
    "KUW": "쿠웨이트",
    "BHR": "바레인",
    "OMA": "오만",
    "YEM": "예멘",
    "SYR": "시리아",
    "LBN": "레바논",
    "JOR": "요르단",
    "ISR": "이스라엘",
    "PSE": "팔레스타인",
    "IRQ": "이라크",
    "IRN": "이란",
    "EGY": "이집트",
    "TUR": "튀르키예"
}

middle_east_events = df[
    df["Actor1CountryCode"].isin(middle_east_cameo) &
    df["Actor2CountryCode"].isin(middle_east_cameo)
].copy()

print("Actor1과 Actor2가 모두 중동 16개국인 이벤트 수:", len(middle_east_events))
middle_east_events.head()

Actor1과 Actor2가 모두 중동 16개국인 이벤트 수: 1916


,GLOBALEVENTID,SQLDATE,MonthYear,Year,FractionDate,Actor1Code,Actor1Name,Actor1CountryCode,Actor1KnownGroupCode,Actor1EthnicCode,...,Actor2Geo_FeatureID,ActionGeo_Type,ActionGeo_FullName,ActionGeo_CountryCode,ActionGeo_ADM1Code,ActionGeo_Lat,ActionGeo_Long,ActionGeo_FeatureID,DATEADDED,SOURCEURL
310,1322780678,20260913,202609,2026,2026.6932,BHR,BAHRAIN,BHR,NaN,NaN,...,10074674,4,"Tehran, Tehran, Iran",IR,IR26,35.75,51.5148,10074674,20260913,https://www.chicagotribune.com/2026/09/12/peop...
311,1322780679,20260913,202609,2026,2026.6932,BHR,BAHRAIN,BHR,NaN,NaN,...,10074674,4,"Tehran, Tehran, Iran",IR,IR26,35.75,51.5148,10074674,20260913,https://www.chicagotribune.com/2026/09/12/peop...
312,1322780680,20260913,202609,2026,2026.6932,BHR,BAHRAIN,BHR,NaN,NaN,...,10074674,1,Bahrain,BA,BA,26,50.5,BA,20260913,https://www.chicagotribune.com/2026/09/12/peop...
313,1322780681,20260913,202609,2026,2026.6932,BHR,BAHRAIN,BHR,NaN,NaN,...,10074674,4,"Tehran, Tehran, Iran",IR,IR26,35.75,51.5148,10074674,20260913,https://www.chicagotribune.com/2026/09/12/peop...
314,1322780682,20260913,202609,2026,2026.6932,BHR,BAHRAIN,BHR,NaN,NaN,...,10074674,1,Bahrain,BA,BA,26,50.5,BA,20260913,https://www.chicagotribune.com/2026/09/12/peop...


## 3. 지정 EventCode 20개 필터링

In [4]:
target_event_codes = [
    "138", "1381", "1382", "1383", "1384", "1385", "139",
    "152", "154", "163",
    "190", "191", "192", "193", "194", "195", "196",
    "204", "2041", "2042"
]

filtered = middle_east_events[
    middle_east_events["EventCode"].isin(target_event_codes)
].copy()

print("선택 EventCode에 해당하는 최종 이벤트 수:", len(filtered))
filtered["EventCode"].value_counts().sort_index()

선택 EventCode에 해당하는 최종 이벤트 수: 379


EventCode
138     22
154      6
163      2
190    228
192     24
193     23
194     36
195     35
196      3
Name: count, dtype: int64

## 4. 국가명과 EventCode 필터링

- EventCode 중 분석 대상
```
    138    군사력 사용 위협
    1381   봉쇄 위협
    1382   점령 위협
    1383   비정규 폭력 사용 위협
    1384   재래식 군사 공격 위협
    1385   대량살상무기 공격 위협
    139    최후통첩
    
    152    군 경계태세 강화
    154    군 병력 동원·증강

    163    금수·보이콧·제재 부과

    190    재래식 군사력 사용
    191    봉쇄·이동 제한
    192    영토 점령
    193    소형무기·경무기를 이용한 전투
    194    포병·전차를 이용한 전투
    195    공중무기 사용
    196    휴전 위반

    204    대량살상무기 사용
    2041   화학·생물학·방사능 무기 사용
    2042   핵무기 폭발
```

- 컬럼 중 필요한 컬럼
    - (제가 생각하기에 핵심적으로 필요하다고 생각해서 추린 것으로 추후 얼마든지 수정 가능합니다)
```
    GLOBALEVENTID         이벤트 고유식별번호
    SQLDATE               이벤트 발생날짜

    Actor1Name            행동 주체의 이름
    Actor1CountryCode     actor1이 속한 국가의 CAMEO 국가코드

    Actor2Name            actor1의 행동이 향한 대상의 이름
    Actor2CountryCode     actor2가 속한 국가의 CAMEO 국가코드

    EventCode             CAMEO 세부 사건 코드
    EventRootCode         EventCode의 상위 대분류코드
    QuadClass             CAMEO 사건을 4개로 분류 (1 언어적 협력 / 2 물질적 협력 / 3 언어적 갈등 / 4 물질적 갈등)

    GoldsteinScale        해당 사건 유형의 협력/갈등 정도를 나타내는 goldstein 점수
    NumMentions           해당 이벤트가 GDELT 데이터에서 언급된 횟수(사건의 언론 노출, 주목도 참고)
    NumSources            해당 이벤트를 보도한 서로 다른 뉴스 출처 수
    NumArticles           해당 이벤트와 연결된 기사 수
    AvgTone               관련 기사들의 평균적인 보도 논조(긍정, 부정)

    ActionGeo_FullName    해당 행동이 발생한 장소 이름
    ActionGeo_CountryCode 행동 발생 장소가 속한 국가코드
    SOURCEURL             해당 이벤트와 연결된 원문 뉴스 기사 URL
```

In [8]:
# 분석 대상 EventCode
target_event_codes = [
    "138",   
    "1381", 
    "1382",  
    "1383",  
    "1384",  
    "1385",  
    "139",  

    "152", 
    "154",  

    "163", 

    "190",  
    "191", 
    "192",  
    "193",
    "194",
    "195",
    "196", 

    "204", 
    "2041",
    "2042" 
]

# EventCode 필터링
filtered = middle_east_events[
    middle_east_events["EventCode"].isin(target_event_codes)
].copy()

# 지정 국가 코드 필터링
filtered["Actor1Country_KR"] = filtered["Actor1CountryCode"].map(middle_east_cameo)
filtered["Actor2Country_KR"] = filtered["Actor2CountryCode"].map(middle_east_cameo)

# actor1과 actor2가 같은 경우도 배체하려면 아래 코드 사용하면 될 것 같습니다.
# middle_east_events = df[
#     df["Actor1CountryCode"].isin(middle_east_cameo) &
#     df["Actor2CountryCode"].isin(middle_east_cameo) &
#     (df["Actor1CountryCode"] != df["Actor2CountryCode"])
# ].copy()


# 분석에 사용할 컬럼
show_cols = [
    "GLOBALEVENTID",
    "SQLDATE",

    "Actor1Name",
    "Actor1CountryCode",

    "Actor2Name",
    "Actor2CountryCode",

    "EventCode",
    "EventRootCode",
    "QuadClass",

    "GoldsteinScale",
    "NumMentions",
    "NumSources",
    "NumArticles",
    "AvgTone",

    "ActionGeo_FullName",
    "ActionGeo_CountryCode",
    "SOURCEURL"
]


filtered[show_cols].head(10)

,GLOBALEVENTID,SQLDATE,Actor1Name,Actor1CountryCode,Actor2Name,Actor2CountryCode,EventCode,EventRootCode,QuadClass,GoldsteinScale,NumMentions,NumSources,NumArticles,AvgTone,ActionGeo_FullName,ActionGeo_CountryCode,SOURCEURL
2086,1322782595,20260913,IRAQ,IRQ,SAUDI,SAU,190,19,4,-10.0,10,1,10,-7.7120822622108,Saudi Arabia,SA,http://www.sitkasentinel.com/stories/iraq-race...
2111,1322782620,20260913,ISRAEL,ISR,TEHRAN,IRN,190,19,4,-10.0,28,5,28,-3.10146030096526,Israel,IS,https://www.freemalaysiatoday.com/category/hig...
2317,1322782826,20260913,HAMAS,PSE,ISRAELI,ISR,190,19,4,-10.0,12,2,12,-5.73009614719513,"Gaza, Israel (general), Israel",IS,https://www.dailykos.com/stories/2026/9/12/800...
2941,1322783930,20260913,IRAQI,IRQ,SAUDI ARABIA,SAU,190,19,4,-10.0,4,1,4,-8.3916083916084,"Riyadh, Ar Riya?, Saudi Arabia",SA,http://thepeninsulaqatar.com/article/12/09/202...
2942,1322783931,20260913,IRAQ,IRQ,SAUDI ARABIA,SAU,190,19,4,-10.0,1,1,1,-8.3916083916084,"Ankara, Ankara, Turkey",TU,http://thepeninsulaqatar.com/article/12/09/202...
2947,1322783936,20260913,ISRAEL,ISR,PALESTINIAN,PSE,190,19,4,-10.0,224,34,205,-6.05782779323015,"Gaza, Israel (general), Israel",IS,https://www.al-monitor.com/originals/2026/09/i...
2948,1322783937,20260913,ISRAEL,ISR,PALESTINIAN,PSE,190,19,4,-10.0,2,2,2,-1.05949323899653,"Gaza, Israel (general), Israel",IS,https://www.al-monitor.com/originals/2026/09/i...
2950,1322783939,20260913,ISRAEL,ISR,HAMAS,PSE,190,19,4,-10.0,12,1,10,-4.8235294117647,"Gaza, Israel (general), Israel",IS,https://www.timeslive.co.za/opinion/2026-09-12...
3818,1322785123,20260913,SAUDI,SAU,YEMEN,YEM,190,19,4,-10.0,8,1,8,-4.43349753694581,"Riyadh, Ar Riya?, Saudi Arabia",SA,https://www.scmp.com/news/world/middle-east/ar...
4206,1322785879,20260913,ISRAELI,ISR,PALESTINIAN,PSE,190,19,4,-10.0,1,1,1,-9.53984287317621,"Zatara, West Bank (general), West Bank",WE,https://imemc.org/article/israeli-forces-colon...


## 5. 국가쌍별 이벤트 수

여기서는 GDELT의 방향성을 유지합니다.

- `Actor1` = 행동 주체
- `Actor2` = 행동 대상

따라서 `ISR → IRN`과 `IRN → ISR`은 서로 다른 국가쌍으로 집계합니다.

+ 해당 국가쌍은 Actor1CountryCode + Actor2CountryCode를 기준으로 한 단순한 기준입니다.

In [9]:
pair_counts = (
    filtered
    .groupby([
        "Actor1CountryCode",
        "Actor2CountryCode"
    ])
    .size()
    .reset_index(name="event_count")
    .sort_values("event_count", ascending=False)
    .reset_index(drop=True)
)

pair_counts.head(30)

,Actor1CountryCode,Actor2CountryCode,event_count
0,ISR,PSE,57
1,YEM,SAU,45
2,SAU,YEM,34
3,ISR,ISR,22
4,IRN,IRN,22
5,PSE,ISR,18
6,SAU,IRQ,17
7,IRQ,SAU,15
8,ISR,SYR,14
9,YEM,YEM,14


## 6. 국가쌍 × EventCode별 이벤트 수 확인

### 6-1. 단순 국가쌍 확인

In [11]:
pair_event_counts = (
    filtered
    .groupby([
        "Actor1CountryCode", "Actor1Country_KR",
        "Actor2CountryCode", "Actor2Country_KR",
        "EventCode"
    ])
    .size()
    .reset_index(name="event_count")
    .sort_values("event_count", ascending=False)
    .reset_index(drop=True)
)

pair_event_counts.head(50)

,Actor1CountryCode,Actor1Country_KR,Actor2CountryCode,Actor2Country_KR,EventCode,event_count
0,ISR,이스라엘,PSE,팔레스타인,190,33
1,YEM,예멘,SAU,사우디아라비아,190,26
2,SAU,사우디아라비아,YEM,예멘,190,18
3,IRN,이란,IRN,이란,190,17
4,PSE,팔레스타인,ISR,이스라엘,190,16
5,IRQ,이라크,SAU,사우디아라비아,190,15
6,ISR,이스라엘,ISR,이스라엘,190,14
7,YEM,예멘,SAU,사우디아라비아,194,11
8,ISR,이스라엘,PSE,팔레스타인,193,11
9,SAU,사우디아라비아,IRQ,이라크,194,9


### 6-2. 국가쌍별 QuadClass 3 / 4 이벤트 수 집계

In [15]:
pair_quad_counts = (
    filtered
    .groupby([
        "Actor1CountryCode", "Actor1Country_KR",
        "Actor2CountryCode", "Actor2Country_KR",
        "QuadClass"
    ])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# 컬럼명 변경
pair_quad_counts = pair_quad_counts.rename(columns={
    "3": "QuadClass_3_count",
    "4": "QuadClass_4_count"
})

# 전체 이벤트 수도 추가
pair_quad_counts["total_event_count"] = (
    pair_quad_counts["QuadClass_3_count"] +
    pair_quad_counts["QuadClass_4_count"]
)

# 전체 이벤트 수가 많은 국가쌍부터 정렬
pair_quad_counts = (
    pair_quad_counts
    .sort_values("total_event_count", ascending=False)
    .reset_index(drop=True)
)

pair_quad_counts.head(30)

QuadClass,Actor1CountryCode,Actor1Country_KR,Actor2CountryCode,Actor2Country_KR,QuadClass_3_count,QuadClass_4_count,total_event_count
0,ISR,이스라엘,PSE,팔레스타인,0,57,57
1,YEM,예멘,SAU,사우디아라비아,1,44,45
2,SAU,사우디아라비아,YEM,예멘,9,25,34
3,ISR,이스라엘,ISR,이스라엘,0,22,22
4,IRN,이란,IRN,이란,0,22,22
5,PSE,팔레스타인,ISR,이스라엘,0,18,18
6,SAU,사우디아라비아,IRQ,이라크,2,15,17
7,IRQ,이라크,SAU,사우디아라비아,0,15,15
8,ISR,이스라엘,SYR,시리아,0,14,14
9,YEM,예멘,YEM,예멘,0,14,14


## 7. 분석용 CSV 저장

In [ ]:
filtered[show_cols].to_csv(
    "GDELT_20260913_중동_선택EventCode_필터링.csv",
    index=False,
    encoding="utf-8-sig"
)

pair_counts.to_csv(
    "GDELT_20260913_중동_국가쌍별_이벤트수.csv",
    index=False,
    encoding="utf-8-sig"
)

pair_event_counts.to_csv(
    "GDELT_20260913_중동_국가쌍_EventCode별_이벤트수.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")